# Trabajo Práctico N°2 — EDA: Análisis Exploratorio de Datos

**Alumno:** Gonzalo Zarazaga

---

Este notebook realiza el Análisis Exploratorio de Datos del dataset *Body Signal of Smoking*.
El objetivo es comprender la distribución de las variables, identificar relaciones con la variable objetivo (`smoking`),
detectar variables irrelevantes y formular hipótesis que guíen el preprocesamiento y el modelado.

**Insumo:** `data/raw/smoking_prediction.csv`  
**Salida:** decisiones documentadas para el notebook `03_preprocesamiento.ipynb`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np

df = pd.read_csv("../data/raw/smoking_prediction.csv")
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head(3)

## 1. Hipótesis de análisis

Las siguientes hipótesis guían el análisis exploratorio del dataset:

- **H1 — Género y tabaquismo:** Los hombres tienen una tasa de tabaquismo significativamente mayor que las mujeres.
  Se espera que `gender` sea el predictor más relevante del modelo, por encima de las variables numéricas.

- **H2 — Sarro y tabaquismo:** La presencia de sarro (`tartar`) está asociada al hábito de fumar.
  El calor, los depósitos de alquitrán y los cambios en la saliva de los fumadores favorecen la acumulación de sarro.

- **H3 — GTP elevada en fumadores:** Los fumadores presentan niveles más altos de GTP (gamma-glutamil transferasa),
  enzima hepática sensible al tabaco y al alcohol.

- **H4 — Hemoglobina elevada en fumadores:** El monóxido de carbono del tabaco desplaza oxígeno de la hemoglobina,
  lo que desencadena una respuesta compensatoria que eleva su concentración en sangre.

## 2. Variable objetivo: `smoking`

La distribución de la variable a predecir es el primer dato a observar: determina el **benchmark del modelo**.
Un clasificador que siempre prediga la clase mayoritaria alcanzaría la exactitud de ese porcentaje sin aprender nada.
El modelo debe superar ese umbral para aportar valor real.

In [ ]:
conteo = df["smoking"].value_counts().sort_index()
proporciones = df["smoking"].value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(7, 5))

colores = ["#3498db", "#e74c3c"]
bars = ax.bar(
    ["No fumador (0)", "Fumador (1)"],
    conteo.values,
    color=colores,
    edgecolor="white",
    width=0.5
)

for bar, prop in zip(bars, proporciones.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 300,
        f"{int(bar.get_height()):,}\n({prop:.1f}%)",
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )

ax.set_title(
    "Distribución de la Variable Objetivo (smoking)\n"
    "Benchmark: un clasificador trivial alcanzaría ~63.3% de exactitud",
    fontsize=12, fontweight="bold"
)
ax.set_ylabel("Cantidad de registros", fontsize=11)
ax.set_ylim(0, conteo.max() * 1.25)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"No fumadores (0): {conteo[0]:,}  →  {proporciones[0]:.1f}%")
print(f"Fumadores    (1): {conteo[1]:,}  →  {proporciones[1]:.1f}%")
print(f"Ratio de desbalance: {conteo[0]/conteo[1]:.2f}:1")
print("\nClase mayoritaria: 0 (No fumador)")
print("El dataset está relativamente balanceado (63/37). No se requiere técnica de re-muestreo.")

## 3. Verificación de duplicados

Se verifica si existen registros idénticos en el dataset. Los duplicados pueden sesgar el modelo
si el mismo paciente aparece múltiples veces con idénticos valores.

In [ ]:
n_total = len(df)
n_duplicados = df.duplicated().sum()

print(f"Total de filas:   {n_total:,}")
print(f"Filas duplicadas: {n_duplicados}")
print(f"Filas únicas:     {n_total - n_duplicados:,}")

if n_duplicados == 0:
    print("\n→ Sin duplicados. No se requiere acción.")
else:
    print(f"\n→ Se detectaron {n_duplicados} filas duplicadas. Revisar antes de continuar.")

## 4. Variables constantes (sin varianza)

Una variable con un único valor posible no aporta información discriminante al modelo:
su valor es idéntico para fumadores y no fumadores, por lo que no puede separar las clases.
Incluirla solo agrega ruido.

Se detectan a través de `nunique() == 1`.

In [ ]:
columnas_constantes = [col for col in df.columns if df[col].nunique() <= 1]

print("Variables con un único valor único (sin varianza):")
for col in columnas_constantes:
    valor_unico = df[col].unique()[0]
    print(f"  → '{col}': único valor = '{valor_unico}'  (dtype: {df[col].dtype})")
    print(f"     Consecuencia: igual para fumadores y no fumadores → candidata a eliminar en preprocesamiento\n")

if not columnas_constantes:
    print("  (ninguna detectada)")

# Confirmación visual
print(f"\nVerificación cruzada de 'oral' con smoking:")
print(df.groupby("smoking")["oral"].value_counts())

## 5. Análisis de variables categóricas

Se analizan las variables categóricas del dataset (`gender`, `tartar`, `dental caries`) en relación con `smoking`.
Para cada una se calcula la **tasa de fumadores** dentro de cada categoría, que es más informativa que
la cantidad absoluta cuando los grupos no están balanceados en tamaño.

In [ ]:
# H1: distribución de fumadores por género
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel izquierdo: conteo absoluto ---
gender_counts = df.groupby(["gender", "smoking"]).size().unstack(fill_value=0)
gender_counts.columns = ["No fumador (0)", "Fumador (1)"]
gender_counts.index = ["Femenino (F)", "Masculino (M)"]

gender_counts.plot(
    kind="bar", ax=axes[0],
    color=["#3498db", "#e74c3c"],
    edgecolor="white", rot=0, width=0.6
)
axes[0].set_title("Distribución absoluta por género", fontsize=11, fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Registros", fontsize=10)
axes[0].legend(fontsize=10)
axes[0].grid(axis="y", alpha=0.3)

# --- Panel derecho: tasa de fumadores por género ---
tasa_por_genero = df.groupby("gender")["smoking"].mean() * 100
tasa_por_genero.index = ["Femenino (F)", "Masculino (M)"]

bars = axes[1].bar(
    tasa_por_genero.index,
    tasa_por_genero.values,
    color=["#e91e8c", "#2196F3"],
    edgecolor="white", width=0.5
)
for bar in bars:
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{bar.get_height():.1f}%",
        ha="center", va="bottom", fontsize=13, fontweight="bold"
    )

axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title("Tasa de fumadores por género", fontsize=11, fontweight="bold")
axes[1].set_ylim(0, 100)
axes[1].grid(axis="y", alpha=0.3)
media_global = df["smoking"].mean() * 100
axes[1].axhline(y=media_global, color="gray", linestyle="--", linewidth=1.2,
                label=f"Media global: {media_global:.1f}%")
axes[1].legend(fontsize=9)

plt.suptitle(
    "H1: Género vs Tabaquismo — ¿Los hombres fuman significativamente más?",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

print("Tasa de fumadores por género:")
for genero, tasa in tasa_por_genero.items():
    print(f"  {genero}: {tasa:.1f}%")

In [ ]:
# H2: distribución de fumadores según presencia de sarro
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel izquierdo: conteo absoluto ---
tartar_counts = df.groupby(["tartar", "smoking"]).size().unstack(fill_value=0)
tartar_counts.columns = ["No fumador (0)", "Fumador (1)"]
tartar_counts.index = ["Sin sarro (N)", "Con sarro (Y)"]

tartar_counts.plot(
    kind="bar", ax=axes[0],
    color=["#3498db", "#e74c3c"],
    edgecolor="white", rot=0, width=0.6
)
axes[0].set_title("Distribución absoluta según presencia de sarro", fontsize=11, fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Registros", fontsize=10)
axes[0].legend(fontsize=10)
axes[0].grid(axis="y", alpha=0.3)

# --- Panel derecho: tasa ---
tasa_por_tartar = df.groupby("tartar")["smoking"].mean() * 100
tasa_por_tartar.index = ["Sin sarro (N)", "Con sarro (Y)"]

bars = axes[1].bar(
    tasa_por_tartar.index,
    tasa_por_tartar.values,
    color=["#27ae60", "#f39c12"],
    edgecolor="white", width=0.5
)
for bar in bars:
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{bar.get_height():.1f}%",
        ha="center", va="bottom", fontsize=13, fontweight="bold"
    )

axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title("Tasa de fumadores según presencia de sarro", fontsize=11, fontweight="bold")
axes[1].set_ylim(0, 80)
axes[1].grid(axis="y", alpha=0.3)
axes[1].axhline(y=media_global, color="gray", linestyle="--", linewidth=1.2,
                label=f"Media global: {media_global:.1f}%")
axes[1].legend(fontsize=9)

plt.suptitle(
    "H2: Sarro vs Tabaquismo — ¿El sarro dental es indicador de fumador?",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

print("Tasa de fumadores según sarro:")
for grupo, tasa in tasa_por_tartar.items():
    print(f"  {grupo}: {tasa:.1f}%")

In [ ]:
# Caries dentales vs smoking
fig, ax = plt.subplots(figsize=(8, 5))

tasa_por_caries = df.groupby("dental caries")["smoking"].mean() * 100
tasa_por_caries.index = ["Sin caries (0)", "Con caries (1)"]

bars = ax.bar(
    tasa_por_caries.index,
    tasa_por_caries.values,
    color=["#27ae60", "#9b59b6"],
    edgecolor="white", width=0.5
)
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{bar.get_height():.1f}%",
        ha="center", va="bottom", fontsize=13, fontweight="bold"
    )

ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(
    "Tasa de fumadores según presencia de caries dentales\n"
    "¿Las caries están asociadas al tabaquismo?",
    fontsize=12, fontweight="bold"
)
ax.set_ylim(0, 60)
ax.grid(axis="y", alpha=0.3)
ax.axhline(y=media_global, color="gray", linestyle="--", linewidth=1.2,
           label=f"Media global: {media_global:.1f}%")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("Tasa de fumadores según caries:")
for grupo, tasa in tasa_por_caries.items():
    print(f"  {grupo}: {tasa:.1f}%")

## 6. Distribución de variables numéricas clave

Se analizan las distribuciones de las variables numéricas con mayor potencial discriminativo según dominio médico,
segmentadas por la variable objetivo.
Se usan histogramas con KDE para visualizar la separación entre fumadores (`smoking=1`) y no fumadores (`smoking=0`).

Las variables seleccionadas son aquellas cuya relación con el tabaco tiene respaldo clínico:
- `hemoglobin`: el CO del tabaco estimula la producción de hemoglobina
- `Gtp`: enzima hepática sensible al tabaco
- `ALT`: otra enzima hepática afectada por el tabaco
- `triglyceride`: los fumadores tienden a tener triglicéridos más altos
- `HDL`: el tabaco reduce el colesterol bueno
- `systolic`: la nicotina eleva la presión arterial

In [ ]:
variables_clave = [
    ("hemoglobin",   "H4: Hemoglobina"),
    ("Gtp",          "H3: GTP — enzima hepática"),
    ("ALT",          "Enzima hepática ALT"),
    ("triglyceride", "Triglicéridos"),
    ("HDL",          "Colesterol HDL (bueno)"),
    ("systolic",     "Presión sistólica")
]

palette = {0: "#3498db", 1: "#e74c3c"}
labels  = {0: "No fumador (0)", 1: "Fumador (1)"}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (col, titulo) in enumerate(variables_clave):
    ax = axes[i]
    for grupo in [0, 1]:
        datos = df[df["smoking"] == grupo][col]
        sns.histplot(
            datos, ax=ax,
            color=palette[grupo],
            label=labels[grupo],
            kde=True, alpha=0.45,
            bins=35, stat="density"
        )

    mediana_0 = df[df["smoking"] == 0][col].median()
    mediana_1 = df[df["smoking"] == 1][col].median()
    ax.axvline(mediana_0, color=palette[0], linestyle="--", linewidth=1.5)
    ax.axvline(mediana_1, color=palette[1], linestyle="--", linewidth=1.5)

    ax.set_title(
        f"{titulo}\nMediana: {mediana_0:.3f} (no fum.) vs {mediana_1:.3f} (fum.)",
        fontsize=10, fontweight="bold"
    )
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel("Densidad", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle(
    "Distribución de Variables Biomédicas Clave: Fumadores vs No Fumadores\n"
    "Líneas punteadas = mediana de cada grupo",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

## 7. Comparación de variables numéricas por grupo (boxplots)

Los boxplots permiten comparar la dispersión y los valores centrales entre fumadores y no fumadores,
incluyendo la visualización de outliers por grupo.
Son complementarios a los histogramas: donde el histograma muestra la forma de la distribución,
el boxplot facilita la comparación de medianas y rangos intercuartílicos.

In [ ]:
variables_boxplot = [
    "hemoglobin", "Gtp", "ALT", "AST", "triglyceride",
    "HDL", "systolic", "relaxation", "weight(kg)", "waist(cm)"
]

palette_box = {0: "#3498db", 1: "#e74c3c"}

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(variables_boxplot):
    ax = axes[i]
    sns.boxplot(
        data=df, x="smoking", y=col,
        palette=palette_box, hue="smoking",
        legend=False, width=0.5, ax=ax,
        flierprops={"marker": "o", "markersize": 1.5, "alpha": 0.2}
    )
    ax.set_title(col, fontsize=10, fontweight="bold")
    ax.set_xlabel("smoking (0=no / 1=sí)", fontsize=8)
    ax.set_ylabel("", fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "Boxplots: Variables Numéricas por Condición de Fumador\n"
    "Comparación de distribuciones, medianas y outliers",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

## 8. Correlación de variables con la variable objetivo

Se cuantifica la relación lineal de cada variable con `smoking` mediante la **correlación de Pearson**.
Las variables categóricas (`gender`, `tartar`) se codifican como binarias antes del cálculo.
La variable `oral` se excluye por ser constante (un único valor).

> **Nota:** La correlación de Pearson solo captura relaciones lineales. Una variable con correlación
> baja puede igualmente ser informativa para modelos no lineales como XGBoost.

In [ ]:
# Codificación binaria de variables categóricas para análisis de correlación
df_encoded = df.copy()
df_encoded["gender_M"]  = (df_encoded["gender"]  == "M").astype(int)  # M=1, F=0
df_encoded["tartar_Y"]  = (df_encoded["tartar"]  == "Y").astype(int)  # Y=1, N=0

# Variables para el análisis (excluye ID, smoking y oral constante)
columnas_analisis = [
    col for col in df_encoded.select_dtypes(include="number").columns
    if col not in ["ID", "smoking"]
] + ["gender_M", "tartar_Y"]

correlaciones = (
    df_encoded[columnas_analisis + ["smoking"]]
    .corr()["smoking"]
    .drop("smoking")
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 9))

colores_bar = ["#e74c3c" if c > 0 else "#3498db" for c in correlaciones.values]
ax.barh(
    correlaciones.index,
    correlaciones.values,
    color=colores_bar,
    edgecolor="white",
    height=0.7
)

ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación de Pearson con smoking", fontsize=11)
ax.set_title(
    "Correlación de cada variable con la variable objetivo (smoking)\n"
    "Rojo = correlación positiva (más en fumadores) · Azul = negativa (menos en fumadores)",
    fontsize=12, fontweight="bold"
)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 5 variables con mayor correlación positiva con smoking:")
print(correlaciones.head(5).to_string())
print("\nTop 5 variables con correlación negativa con smoking:")
print(correlaciones.tail(5).to_string())

In [ ]:
# Mapa de correlaciones completo entre todas las variables
df_heatmap = df_encoded[columnas_analisis + ["smoking"]].copy()
df_heatmap = df_heatmap.rename(columns={
    "gender_M": "gender (M=1)",
    "tartar_Y": "tartar (Y=1)"
})

fig, ax = plt.subplots(figsize=(18, 15))

corr_matrix = df_heatmap.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    annot=True, fmt=".2f",
    cmap="coolwarm", center=0,
    linewidths=0.3, square=True,
    mask=mask, ax=ax,
    annot_kws={"size": 7}
)

ax.set_title(
    "Mapa de Correlaciones entre Variables\n"
    "(triángulo inferior — valores cercanos a ±1 indican mayor relación lineal)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 9. Conclusiones del EDA

### Variables a eliminar en el preprocesamiento

| Variable | Motivo |
|---|---|
| `oral` | Constante: único valor `"Y"` en todos los registros — sin varianza, sin poder predictivo |
| `ID` | Identificador técnico — no representa información del paciente |

---

### H1 — El género es el predictor más relevante
**Confirmada.** Los hombres presentan una tasa de tabaquismo significativamente mayor que las mujeres.
`gender` (codificada como M=1, F=0) muestra la mayor correlación absoluta con `smoking` entre todas las
variables del dataset. Es la variable más discriminante para el modelo y la única categórica relevante.

---

### H2 — El sarro está asociado al tabaquismo
**Confirmada.** Los pacientes con sarro (`tartar=Y`) presentan una tasa de tabaquismo notablemente más alta
que aquellos sin sarro. La variable `tartar` es un predictor relevante que debe incluirse codificada como binaria.

---

### H3 — GTP elevada en fumadores
**Confirmada.** La distribución de GTP se desplaza claramente hacia valores más altos en fumadores.
La diferencia entre medianas de ambos grupos es visible tanto en el histograma como en el boxplot.

---

### H4 — Hemoglobina elevada en fumadores
**Confirmada.** La distribución de hemoglobina en fumadores se desplaza a la derecha respecto a la de
no fumadores, con una diferencia de medianas apreciable. Es una de las variables numéricas con mayor
correlación con `smoking`.

---

### Hallazgos adicionales
- **HDL más bajo en fumadores:** consistente con la literatura médica — el tabaco reduce el colesterol bueno.
- **Outliers en Gtp, ALT, AST, triglicéridos:** valores extremos presentes, pero no se eliminarán dado que
  XGBoost es robusto a outliers y su eliminación podría sesgar el modelo.
- **Dataset sin nulos ni duplicados:** no se requiere imputación ni deduplicación.
- **Balance aceptable:** ratio 63/37. No se requiere re-muestreo (SMOTE, pesos de clase, etc.).
- **Correlaciones entre features:** `weight(kg)` y `waist(cm)` presentan correlación alta entre sí
  (colinealidad). XGBoost maneja la colinealidad sin problema; no es necesario eliminar ninguna.

---

### Decisiones para el preprocesamiento (`03_preprocesamiento.ipynb`)

1. **Eliminar** columnas `oral` (constante) e `ID` (identificador)
2. **Codificar** `gender` → binaria (M=1, F=0) y `tartar` → binaria (Y=1, N=0)
3. **No imputar** (el dataset no presenta nulos)
4. **Conservar outliers** en variables numéricas (XGBoost es robusto; eliminarlos no es necesario)
5. **Aplicar las mismas transformaciones** al dataset de predicción (`smoking_prediction_entrega.csv`)
6. **Exportar** el dataset procesado a `data/processed/smoking_train_processed.csv`